<a href="https://colab.research.google.com/github/littleadam/AI_GenAI/blob/main/Copy_of_mirae_auto_login_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/littleadam/test_repo.git

In [ ]:
import os
os.chdir('test_repo')

In [ ]:
!python3 config_structure.py

In [ ]:
!pwd

In [ ]:
import requests
import json
import os
from datetime import datetime

CONFIG_FILE = "config.json"

# Load config
def load_config():
    if not os.path.exists(CONFIG_FILE):
        raise FileNotFoundError("Missing config.json file.")
    with open(CONFIG_FILE, 'r') as f:
        return json.load(f)

# Save updated config
def save_config(config):
    with open(CONFIG_FILE, 'w') as f:
        json.dump(config, f, indent=2)

# Check if login is already done today
def is_logged_in_today(config):
    today = datetime.now().strftime("%Y-%m-%d")
    return config.get("last_login_date") == today

def perform_login_and_get_access_token(config):
    print("🔐 Performing authentication...")

    headers = {
        'X-Mirae-Version': '1 ',
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    }

    login_data = {
        'username': config['username'],
        'password': config['password']
    }

    login_response = requests.post(
        'https://api.mstock.trade/openapi/typea/connect/login',
        headers=headers,
        data=login_data
    )

    print(f"Login response: {login_response.text}")
    login_json = login_response.json()
    name = login_json.get("data", {}).get("nm")
    print(f"Logged in user: {name}")

    # Ask for request token
    request_token = input("🔑 Enter request_token: ").strip()

    token_data = {
        'api_key': config['api_key'],
        'request_token': request_token,
        'checksum': 'I'  # This might need to be calculated based on mStock API documentation
    }

    token_response = requests.post(
        'https://api.mstock.trade/openapi/typea/session/token',
        headers=headers,
        data=token_data
    )

    print(f"Token response: {token_response.text}")

    try:
        token_json = token_response.json()
        data_obj = token_json.get("data", {})

        login_time = data_obj.get("login_time")
        access_token = (
            token_json.get("access_token") or
            token_json.get("accessToken") or
            token_json.get("token") or
            data_obj.get("access_token") or
            data_obj.get("accessToken") or
            data_obj.get("token")
        )

        print(f"✅ Login Time: {login_time}")
        print(f"✅ Access Token: {access_token}")

        if access_token:
            config["access_token"] = access_token
            config["last_login_date"] = datetime.now().strftime("%Y-%m-%d")
            save_config(config)
        else:
            print("❌ Access token not found in response.")

    except Exception as e:
        print(f"❌ Error parsing token response: {e}")
        print("Raw response text:", token_response.text)

# ---------- Main Execution ----------

try:
    config = load_config()

    if is_logged_in_today(config):
        print("✅ Already logged in today. Skipping authentication.")
        print(f"Access Token: {config.get('access_token')}")
    else:
        perform_login_and_get_access_token(config)

except Exception as e:
    print(f"⚠️ Error: {e}")


In [ ]:
#!python3 broker_api.py
%%writefile config.json
{
  "username": "8056023186",
  "password": "Inba@24822!",
  "api_key": "JDn9PAVBBMRWRWsZ2Pvo1g==",
  "last_login_date": "2025-07-27",
  "access_token": "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJVU0VSTkFNRSI6Ik1BMTIzNTUiLCJDTElFTlROQU1FIjoiQVJVTE1VUlVHQSIsIlVTRVJfREVUQUlMUyI6IndvaGduQXZPVlROTXZvYTJIeU4vbmk2ckFzb2lFYUlVTDlLd25HeVRBb05TemJZTU1zN2hEREl1QkpFK2Zad0R5TjBrTTJDWmFIK3JUOS9lbE15RDB5eCt3cEpMQUFrSnArWEVhYWM1UEdISXFFR3RuWXNKSlZzNnJCN2trNnpxQ1Y1eWhIU3FydjJORkNIeUlZL1FQWXRPT3hoMTVLTWZXUWNoS1dleGFwdHBoY1hjVkVjNzFST0RoQkozMmIzTS93cENNWkpUMjhsNWZ6VTJWV0Vwc1NwaGdhS0VOaTFBU2pYU0hvNkRnb1BVdFUveUZjaHB6cGd1R1kzZ0VnZlpUZXVzdXVqcXlWMkFUQ2lnNlI3WmR1cmF4aUJGYWpDRmpmRmtlaDhKdFVTZnp4S0paVE53Ymp5eE9DTnF6R0l0eGRoWEl0VU85V3p6VDdPakZ5RkhFRzArRWFlMXVteWRtM05HRHBhMFBDVT0iLCJVU0VSSUQiOiJNQTEyMzU1IiwiQUNDRVNTX1RPS0VOIjoiZXlKaGJHY2lPaUpJVXpJMU5pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SmhkV1FpT2lKdGFYSmhaUzVwYmlJc0ltVjRjQ0k2TVRjMU16Y3dOVEExTWl3aWFXRjBJam94TnpVek5qRTROalV5TENKcGMzTWlPaUp0YVhKaFpTNXBiaUlzSW01aVppSTZNVFEwTkRRM09EUXdNQ3dpY0dadElqb2lNU0lzSW5ScFpDSTZJalV6SWl3aWRXbGtJam9pTkRJek9EY2lMQ0oyYVdRaU9pSXlNU0o5LnMzcFh1Z2JUVVhETGNGV2JWak1TTzFPOVhQb3FfSF9XZl8zYzl6VmpJUjQiLCJBUElUWVBFIjoiVFlQRUEiLCJVSUQiOiIwZmIxYWZlNy1lMGQwLTRhNmQtOGIxYS03NGI2Nzk2Y2FmMjYiLCJuYmYiOjE3NTM2MTg2NTMsImV4cCI6MTc1MzY0MTAwMCwiaWF0IjoxNzUzNjE4NjUzfQ.dmKpveV_pz3APUcARW6SQNEdD4691jFWp8di5zlqXfA"
}

Overwriting config.json


In [ ]:
import requests
import json
import os
from datetime import datetime

CONFIG_FILE = "config.json"

# Load config
def load_config():
    if not os.path.exists(CONFIG_FILE):
        raise FileNotFoundError("Missing config.json file.")
    with open(CONFIG_FILE, 'r') as f:
        return json.load(f)

# Save updated config
def save_config(config):
    with open(CONFIG_FILE, 'w') as f:
        json.dump(config, f, indent=2)

# Check if login is already done today
def is_logged_in_today(config):
    today = datetime.now().strftime("%Y-%m-%d")
    return config.get("last_login_date") == today

# Perform login and get access token
def perform_login_and_get_access_token(config):
    print("🔐 Performing authentication...")

    headers = {
        'X-Mirae-Version': '1 ',
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    }

    login_data = {
        'username': config['username'],
        'password': config['password']
    }

    login_response = requests.post(
        'https://api.mstock.trade/openapi/typea/connect/login',
        headers=headers,
        data=login_data
    )

    print(f"Login response: {login_response.text}")
    login_json = login_response.json()
    name = login_json.get("data", {}).get("nm")
    print(f"Logged in user: {name}")

    # Ask for request token
    request_token = input("🔑 Enter request_token: ").strip()

    token_data = {
        'api_key': config['api_key'],
        'request_token': request_token,
        'checksum': 'I'  # Use actual checksum if required
    }

    token_response = requests.post(
        'https://api.mstock.trade/openapi/typea/session/token',
        headers=headers,
        data=token_data
    )

    print(f"Token response: {token_response.text}")

    try:
        token_json = token_response.json()
        data_obj = token_json.get("data", {})

        login_time = data_obj.get("login_time")
        access_token = (
            token_json.get("access_token") or
            token_json.get("accessToken") or
            token_json.get("token") or
            data_obj.get("access_token") or
            data_obj.get("accessToken") or
            data_obj.get("token")
        )

        print(f"✅ Login Time: {login_time}")
        print(f"✅ Access Token: {access_token}")

        if access_token:
            config["access_token"] = access_token
            config["last_login_date"] = datetime.now().strftime("%Y-%m-%d")
            save_config(config)
        else:
            print("❌ Access token not found in response.")

    except Exception as e:
        print(f"❌ Error parsing token response: {e}")
        print("Raw response text:", token_response.text)

# Fetch funds, holdings, positions
def fetch_account_details(config):
    headers_auth = {
        'X-Mirae-Version': '1',
        'Authorization': f"Bearer {config['access_token']}",
        'X-PrivateKey': config['api_key']
    }

    # 1. Get Fund Summary
    print("\n📊 Fetching Fund Summary...")
    resp_funds = requests.get(
        'https://api.mstock.trade/openapi/typea/user/fundsummary',
        headers=headers_auth
    )
    try:
        fund_data = resp_funds.json().get('data', {})
        print("Funds:", json.dumps(fund_data, indent=2))
    except Exception as e:
        print(f"Error fetching fund summary: {e}")
        print(resp_funds.text)

    # 2. Get Portfolio Holdings
    print("\n📈 Fetching Holdings...")
    resp_holdings = requests.get(
        'https://api.mstock.trade/openapi/typeb/portfolio/holdings',
        headers=headers_auth
    )
    try:
        holdings = resp_holdings.json().get('data', [])
        for h in holdings:
            print(f" - {h.get('tradingsymbol')}: qty={h.get('quantity')}, avg_price={h.get('averageprice')}, ltp={h.get('ltp')}")
    except Exception as e:
        print(f"Error fetching holdings: {e}")
        print(resp_holdings.text)

    # 3. Get Net Positions
    print("\n📉 Fetching Positions...")
    resp_positions = requests.get(
        'https://api.mstock.trade/openapi/typeb/portfolio/positions',
        headers=headers_auth
    )
    try:
        positions = resp_positions.json().get('data', [])
        for p in positions:
            print(f" - {p.get('tradingsymbol') or p.get('symbolname')}: netqty={p.get('netqty')}, netvalue={p.get('netvalue')}")
    except Exception as e:
        print(f"Error fetching positions: {e}")
        print(resp_positions.text)

# --- Main Execution ---
try:
    config = load_config()

    if is_logged_in_today(config):
        print("✅ Already logged in today. Skipping authentication.")
    else:
        perform_login_and_get_access_token(config)

    # Fetch portfolio info
    fetch_account_details(config)

except Exception as e:
    print(f"⚠️ Error: {e}")


🔐 Performing authentication...
Login response: {"status":"success","data":{"ugid":"9db5087c-d475-45e2-9f31-d70729669ed8","is_kyc":true,"is_activate":false,"is_password_reset":true,"is_error":false,"cid":"MA12355","nm":"ARULMURUGA","flag":0}}
Logged in user: ARULMURUGA
🔑 Enter request_token: 382
Token response: {"status":"success","data":{"user_type":"individual","email":"","user_name":"ARULMURUGA","user_shortname":"CDSL","broker":"MIRAE","exchanges":["BEQ","BFO","NEQ","NFO","NCR","BSEMF","NSEIPO","BSEIPO"],"products":["CNC","NRML","MIS"],"order_types":["MARKET","LIMIT","SL","SL-M"],"avatar_url":"","user_id":"12020","api_key":"JDn9PAVBBMRWRWsZ2Pvo1g==","access_token":"eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJVU0VSTkFNRSI6Ik1BMTIzNTUiLCJDTElFTlROQU1FIjoiQVJVTE1VUlVHQSIsIlVTRVJfREVUQUlMUyI6Inh6SkEyTzc5Mm5KQ0R0RHlzWEtWY3gzb1p3YXIxRjFzWmNSK0xQbWlaVVF1UExvZUUrSXJNc1k0cDNuVlVkY3U1bVpjQ1Q0RXVQS2xRS1gzYzlSNnJ6eTFVWnJ2d2xYMVlPZlh1bXl0azZkbmpmbHpMYjk4T2N0YnByS3cwRWIzVnJOSTJZWm5iNUR5M0lrbUt2MGdVUzY

In [ ]:
import requests
import json



def get_auth_headers_type2(config):
    print("headers used:")
    print(f"token{config['api_key']}:{config['access_token']}")
    return {
        'X-Mirae-Version': '1',
        'Authorization': f"token {config['api_key']}:{config['access_token']}"
        #'X-PrivateKey': config['api_key']
    }

def get_auth_headers(config):
    return {
        'X-Mirae-Version': '1',
        'Authorization': f"Bearer {config['access_token']}",
        'X-PrivateKey': config['api_key']
    }

def get_fund_summary(config):
    headers = get_auth_headers_type2(config)
    resp = requests.get('https://api.mstock.trade/openapi/typea/user/fundsummary', headers=headers)
    print("Fund summary is:")
    print(resp.text)
    try:
        return resp.json().get('data', {})
    except Exception:
        print("Error parsing fund summary:", resp.text)
        return {}

def get_holdings(config):
    headers = get_auth_headers(config)
    resp = requests.get('https://api.mstock.trade/openapi/typeb/portfolio/holdings', headers=headers)
    try:
        return resp.json().get('data', [])
    except Exception:
        print("Error parsing holdings:", resp.text)
        return []

def get_positions(config):
    headers = get_auth_headers(config)
    resp = requests.get('https://api.mstock.trade/openapi/typeb/portfolio/positions', headers=headers)
    try:
        return resp.json().get('data', [])
    except Exception:
        print("Error parsing positions:", resp.text)
        return []

def get_ltp(config, instruments):
    print("🟢 get_ltp() called with instruments:", instruments)

    #headers = {
    #    'X-Mirae-Version': '1',
    #    'Authorization': f"Bearer {config['access_token']}",
    #    'X-PrivateKey': config['api_key']
    #}
    headers = get_auth_headers_type2(config)
    params = {
        'i': instruments
    }

    response = requests.get(
        'https://api.mstock.trade/openapi/typea/instruments/quote/ltp',
        params=params,
        headers=headers
    )

    print(f"Response status code: {response.status_code}")
    print(f"Raw response text: {response.text}")

    try:
        data = response.json().get("data", {})
        return data
    except Exception as e:
        print(f"Error fetching LTP: {e}")
        return {}


In [ ]:
import json
import os
from datetime import datetime
import requests
#import broker_api

CONFIG_FILE = "config.json"

def load_config():
    if not os.path.exists(CONFIG_FILE):
        raise FileNotFoundError("Missing config.json file.")
    with open(CONFIG_FILE, 'r') as f:
        return json.load(f)

def save_config(config):
    with open(CONFIG_FILE, 'w') as f:
        json.dump(config, f, indent=2)

def is_logged_in_today(config):
    today = datetime.now().strftime("%Y-%m-%d")
    return config.get("last_login_date") == today

def perform_login_and_get_access_token(config):
    print("🔐 Performing authentication...")

    headers = {
        'X-Mirae-Version': '1 ',
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    }

    login_data = {
        'username': config['username'],
        'password': config['password']
    }

    login_response = requests.post(
        'https://api.mstock.trade/openapi/typea/connect/login',
        headers=headers,
        data=login_data
    )

    print(f"Login response: {login_response.text}")
    login_json = login_response.json()
    name = login_json.get("data", {}).get("nm")
    print(f"Logged in user: {name}")

    request_token = input("🔑 Enter request_token: ").strip()

    token_data = {
        'api_key': config['api_key'],
        'request_token': request_token,
        'checksum': 'I'
    }

    token_response = requests.post(
        'https://api.mstock.trade/openapi/typea/session/token',
        headers=headers,
        data=token_data
    )

    print(f"Token response: {token_response.text}")

    try:
        token_json = token_response.json()
        data_obj = token_json.get("data", {})

        login_time = data_obj.get("login_time")
        access_token = (
            token_json.get("access_token") or
            token_json.get("accessToken") or
            token_json.get("token") or
            data_obj.get("access_token") or
            data_obj.get("accessToken") or
            data_obj.get("token")
        )

        print(f"✅ Login Time: {login_time}")
        print(f"✅ Access Token: {access_token}")

        if access_token:
            config["access_token"] = access_token
            config["last_login_date"] = datetime.now().strftime("%Y-%m-%d")
            save_config(config)
        else:
            print("❌ Access token not found in response.")

    except Exception as e:
        print(f"❌ Error parsing token response: {e}")
        print("Raw response text:", token_response.text)

# ---------- Main Execution ----------
try:
    config = load_config()

    if is_logged_in_today(config):
        print("✅ Already logged in today. Skipping authentication.")
    else:
        perform_login_and_get_access_token(config)

    # 1. Fund Summary
    #print("\n💰 Fund Summary:")
    fund_data =get_fund_summary(config)
    #fund_data = get_fund_summary(config)
    print(config)
    print(json.dumps(fund_data, indent=2))

    # 2. Holdings
    #print("\n📈 Holdings:")
    #holdings = broker_api.get_holdings(config)
    #for h in holdings:
    #    print(f" - {h.get('tradingsymbol')}: qty={h.get('quantity')}, avg_price={h.get('averageprice')}, ltp={h.get('ltp')}")

    # 3. Positions
    #print("\n📉 Positions:")
    #positions = broker_api.get_positions(config)
    #for p in positions:
    #    print(f" - {p.get('tradingsymbol') or p.get('symbolname')}: netqty={p.get('netqty')}, netvalue={p.get('netvalue')}")

    # 4. LTPs
    print("\n📊 LTPs:")

    # Replace with actual trading symbols
    #symbols = [
    #    "NSE:ACC",                     # Nifty 50 index
    #    "NFO:CDSL25JAN2220CE"         # Example Option (replace with actual symbol from contract note)
    #]

    #ltp_data = broker_api.get_ltp(config, symbols)
    #print("fetched LTP")
    #for symbol, info in ltp_data.items():
    #    print(f"{symbol} => LTP: {info.get('ltp')}")
    # 4. LTPs
    #print("\n📊 LTPs:")

    symbols = [
      "NSE:NIFTY 50",                     # Replace with actual index symbol
      "NSE:NIFTY-28Aug2025-24200-PE"         # Replace with valid option symbol
    ]

    ltp_data = get_ltp(config, symbols)

    for symbol, info in ltp_data.items():
        print(f"{symbol} => LTP: {info.get('ltp')}")


except Exception as e:
    print(f"⚠️ Error: {e}")


✅ Already logged in today. Skipping authentication.
headers used:
tokenJDn9PAVBBMRWRWsZ2Pvo1g==:eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJVU0VSTkFNRSI6Ik1BMTIzNTUiLCJDTElFTlROQU1FIjoiQVJVTE1VUlVHQSIsIlVTRVJfREVUQUlMUyI6Inh6SkEyTzc5Mm5KQ0R0RHlzWEtWY3gzb1p3YXIxRjFzWmNSK0xQbWlaVVF1UExvZUUrSXJNc1k0cDNuVlVkY3U1bVpjQ1Q0RXVQS2xRS1gzYzlSNnJ6eTFVWnJ2d2xYMVlPZlh1bXl0azZkbmpmbHpMYjk4T2N0YnByS3cwRWIzVnJOSTJZWm5iNUR5M0lrbUt2MGdVUzYvbW5oN1ExczVnN3FOZ2tYR1JUMm1IbGp1RmcyZC91QTJwcVA2S0twYzFiblRTRkI2ZWRIZU4vVWUrUmRvYm83Y1p5TG0vRnJOVFcrdXpCQ2gvMDY5cy81d0FCWEFMSmtjOUhPQ3pDOFNvTUFMU01vWUJvNWZTNEkxcnlnQU83N2syTGNUSzJCMVliM0xhZ2JBbnpVejJnRnl0cVBYRGZGQmtVTEE1S2RXK0cwQkY2UHNhdGo0YjlVN2RZQk9nMDRkUzluSU9veExvVHpRajI5cWxoTT0iLCJVU0VSSUQiOiJNQTEyMzU1IiwiQUNDRVNTX1RPS0VOIjoiZXlKaGJHY2lPaUpJVXpJMU5pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SmhkV1FpT2lKdGFYSmhaUzVwYmlJc0ltVjRjQ0k2TVRjMU5UUTRNakUwTWl3aWFXRjBJam94TnpVMU16azFOelF5TENKcGMzTWlPaUp0YVhKaFpTNXBiaUlzSW01aVppSTZNVFEwTkRRM09EUXdNQ3dpY0dadElqb2lNU0lzSW5ScFpDSTZJamcwSWl3aWRXbGt